# 05 — Cotizaciones de mercado
**Spec:** docs/specs/05_cotizaciones.md · **Requisitos:** R-16 a R-21 · **ADR:** 005, 008, 009
**Escribe:** `iol_bronze.quotes_raw`, `iol_ops.quote_requests`, `iol_ops.quote_series`, `iol_ops.quote_coverage`, `iol_gold.fact_market_quotes`, `iol_gold.fact_trade_market`, y reconstruye `iol_ops.instrument_mapping`

Compara cada operación contra el **cierre diario** de su símbolo y moneda. Es un benchmark de cierre, no el precio al instante: la hora del dataset no es confiable (ADR-003).
Ninguna fila se descarta: una operación sin cotización queda en `fact_trade_market` y `match_method` dice por qué.

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("quotes_source", "api", ["api", "volume"])
dbutils.widgets.dropdown("demo_rerun", "true", ["true", "false"])

import json
import os
import sys
import uuid
from pathlib import Path

import requests
from pyspark.sql import functions as F

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import config, instruments, io, quotes

RUN_ID = dbutils.widgets.get("pipeline_run_id") or str(uuid.uuid4())
CATALOG = dbutils.widgets.get("catalog")
SOURCE = dbutils.widgets.get("quotes_source")
print("pipeline_run_id:", RUN_ID, "· fuente:", SOURCE)

io.ensure_schemas(spark, catalog=CATALOG)
silver_table = io.table_name("silver", "trades", catalog=CATALOG)
mapping_table = io.table_name("ops", "instrument_mapping", catalog=CATALOG)
enrichment_table = io.table_name("ops", "instrument_enrichment", catalog=CATALOG)
raw_table = io.table_name("bronze", "quotes_raw", catalog=CATALOG)
requests_table = io.table_name("ops", "quote_requests", catalog=CATALOG)
series_table = io.table_name("ops", "quote_series", catalog=CATALOG)
coverage_table = io.table_name("ops", "quote_coverage", catalog=CATALOG)
market_table = io.table_name("gold", "fact_market_quotes", catalog=CATALOG)
trade_market_table = io.table_name("gold", "fact_trade_market", catalog=CATALOG)

pipeline_run_id: 0a4f1448-24c4-4027-9da4-693a6ddabe48 · fuente: api


## 1. Descarga
Una fila por símbolo y proveedor de su cadena (ADR-008). Lo ya pedido sin error no se repite: si la corrida se corta, la siguiente sigue desde ahí.
Con `quotes_source = volume` lee lo que `tools/quotes_fetcher` dejó en el Volume, con el mismo contrato.

In [0]:
symbols = [r.asDict() for r in spark.table(mapping_table).select("symbol", "asset_type").collect()]
candidates = quotes.candidate_series(symbols)
names = sorted({c["provider"] for c in candidates})
if SOURCE == "api":
    providers = {name: quotes.build_provider(name, symbols, requests.get) for name in names}
    pause_s = config.QUOTE_PAUSE_S
else:
    providers = {name: quotes.FileReplayProvider(name, Path(io.landing_path("quotes", catalog=CATALOG))) for name in names}
    pause_s = 0.0
print("Series candidatas:", len(candidates), "de", len(symbols), "símbolos")
fetched = quotes.run_fetch(spark, candidates, providers, raw_table, requests_table, SOURCE, RUN_ID, pause_s=pause_s)
print("Descarga:", fetched)
display(quotes.reconcile_requests(spark.table(requests_table), spark.table(raw_table)))

Series candidatas: 2936 de 1445 símbolos
Descarga: {'requested': 5, 'with_rows': 0, 'errors': 5, 'skipped': 2931}


provider,requests,requests_with_rows,requests_with_error,rows_returned,rows_stored
byma_open_data,1445,1146,0,66236,66236
data912,804,261,0,15595,15595
yahoo,687,584,5,33702,33702


## 2. Validación de las series
De cada símbolo se usa la primera serie de su cadena que valida. Nunca la que más se parece a las operaciones: sesgaría la comparación (ADR-008).

In [0]:
trades = spark.table(silver_table)
print("Series:", quotes.run_series(spark, candidates, raw_table, trades, series_table))
print("Mapeo:", instruments.run_instruments(spark, silver_table, mapping_table, enrichment_table=enrichment_table, quote_series_table=series_table))
series = spark.table(series_table)
display(quotes.series_summary(series))
display(spark.table(mapping_table).groupBy("mapping_status").count().orderBy("mapping_status"))
display(series.where("series_status = 'SCALE_SUSPECT'").select("symbol", "provider", "days_compared", "ratio_p50").orderBy("symbol"))

Series: {'rows_inserted': 0, 'rows_updated': 0}
Mapeo: {'rows_inserted': 0, 'rows_updated': 0, 'symbols': 1445}


provider,series_status,series,selected
byma_open_data,BIASED,621,0
byma_open_data,NO_DATA,299,0
byma_open_data,NO_OVERLAP,5,0
byma_open_data,OK,400,96
byma_open_data,SCALE_SUSPECT,11,0
byma_open_data,THIN,109,51
data912,BIASED,8,0
data912,NO_DATA,543,0
data912,OK,232,232
data912,THIN,21,20


mapping_status,count
MAPPED,738
UNSUPPORTED,707


symbol,provider,days_compared,ratio_p50
AER9O,byma_open_data,24,0.020503935342329346
BKNG,yahoo,25,24.96083903394256
BKNG,byma_open_data,25,25.138449068352383
BKNGD,byma_open_data,2,25.66667082917083
CO26,byma_open_data,1,0.03590157690690781
CP28O,byma_open_data,1,0.022080772924370034
HUT,byma_open_data,44,25.09780539292668
HUT,yahoo,44,25.09780539292668
HUTD,byma_open_data,7,24.65579949238579
HUTD,yahoo,7,24.655800128963882


## 3. Benchmark y matching
Misma fecha efectiva → `EXACT_DAY`. Si no, el cierre previo con hasta 3 días hábiles → `PRIOR_DAY_IMPUTED`. Nunca una cotización posterior (ADR-009).

In [0]:
first = quotes.run_market(spark, silver_table, mapping_table, raw_table, requests_table, series_table, market_table, trade_market_table, coverage_table, RUN_ID)
print("1ra corrida:", first)
if dbutils.widgets.get("demo_rerun") == "true":
    second = quotes.run_market(spark, silver_table, mapping_table, raw_table, requests_table, series_table, market_table, trade_market_table, coverage_table, RUN_ID)
    print("2da corrida (tiene que dar 0 en todo):", second)

matched = spark.table(trade_market_table)
trades_count, matched_count = trades.count(), matched.count()
print("Operaciones en Silver:", trades_count, "· en fact_trade_market:", matched_count)
assert trades_count == matched_count, "fact_trade_market tiene que tener una fila por operación"
display(matched.groupBy("match_method").count().orderBy(F.col("count").desc()))

1ra corrida: {'quotes': {'rows_inserted': 0, 'rows_updated': 0}, 'trade_market': {'rows_inserted': 0, 'rows_updated': 0}, 'coverage': {'rows_inserted': 0, 'rows_updated': 5}}
2da corrida (tiene que dar 0 en todo): {'quotes': {'rows_inserted': 0, 'rows_updated': 0}, 'trade_market': {'rows_inserted': 0, 'rows_updated': 0}, 'coverage': {'rows_inserted': 0, 'rows_updated': 0}}
Operaciones en Silver: 100000 · en fact_trade_market: 100000


match_method,count
EXACT_DAY,93733
SOURCE_BIASED,4093
UNAVAILABLE,2051
SCALE_SUSPECT,88
PRIOR_DAY_IMPUTED,35


## 4. Cobertura (R-21)
El denominador son las operaciones elegibles, sin excluir los tipos sin fuente. "Excluidas" son las que tienen serie, pero sesgada o con otra escala.

In [0]:
coverage = spark.table(coverage_table)
overall = quotes.overall_coverage(coverage)
print("Cobertura ponderada por operaciones:", overall)
display(quotes.coverage_by_type(coverage))
display(coverage.groupBy("provider", "series_status").agg(F.count("*").alias("symbols"), F.sum("trades_eligible").alias("eligible")).orderBy(F.col("eligible").desc()))
display(coverage.orderBy(F.col("trades_eligible").desc()).drop("last_error").limit(config.QUOTE_COVERAGE_TOP_N))
display(coverage.where("last_error IS NOT NULL").select("symbol", "asset_type", "provider", "coverage_pct", "last_error").orderBy("symbol"))

Cobertura ponderada por operaciones: {'trades_total': 100000, 'trades_eligible': 99992, 'exact': 93726, 'imputed': 35, 'unavailable': 2051, 'excluded_by_validation': 4180, 'unmapped': 0, 'coverage_pct': 0.9377}


asset_type,symbols,eligible,exact,imputed,unavailable,excluded_by_validation,coverage_pct
CEDEAR,599,54053,53208,24,12,809,0.9848
BONO_SOBERANO,98,18285,17653,3,0,629,0.9656
ACCION_LOCAL,88,18164,18092,0,0,72,0.996
LETRA_LECAP_BONCAP,28,4515,4360,7,0,148,0.9672
ON,315,2550,103,1,22,2424,0.0408
OPCION,295,2016,0,0,2016,0,0.0
BONO_SUBSOBERANO,19,313,215,0,0,98,0.6869
WARRANT,2,95,95,0,0,0,1.0
OTRO,1,1,0,0,1,0,0.0


provider,series_status,symbols,eligible
data912,OK,232,63545
yahoo,OK,265,21075
byma_open_data,OK,96,8796
null,null,707,6229
yahoo,THIN,74,186
byma_open_data,THIN,51,107
data912,THIN,20,54


symbol,currency_code,asset_type,provider,provider_symbol,series_status,price_scale_factor,bias_pct,trades_total,trades_eligible,matched_exact,matched_imputed,unavailable,excluded_by_validation,unmapped,coverage_pct
AL30,ARS,BONO_SOBERANO,data912,AL30,OK,0.01,1.46E-4,8859,8858,8858,0,0,0,0,1.0
AL30D,USD,BONO_SOBERANO,data912,AL30D,OK,0.01,-4.22E-4,4589,4589,4589,0,0,0,0,1.0
MSFT,ARS,CEDEAR,data912,MSFT,OK,1.0,0.001441,2336,2336,2336,0,0,0,0,1.0
SPY,ARS,CEDEAR,data912,SPY,OK,1.0,5.92E-4,2311,2311,2311,0,0,0,0,1.0
AMZN,ARS,CEDEAR,data912,AMZN,OK,1.0,0.001684,2054,2054,2054,0,0,0,0,1.0
MELI,ARS,CEDEAR,data912,MELI,OK,1.0,0.001437,1881,1881,1881,0,0,0,0,1.0
PAMP,ARS,ACCION_LOCAL,data912,PAMP,OK,1.0,0.001656,1668,1668,1668,0,0,0,0,1.0
NVDA,ARS,CEDEAR,data912,NVDA,OK,1.0,0.003102,1493,1493,1493,0,0,0,0,1.0
YPFD,ARS,ACCION_LOCAL,data912,YPFD,OK,1.0,8.84E-4,1485,1485,1485,0,0,0,0,1.0
GOOGL,ARS,CEDEAR,data912,GOOGL,OK,1.0,0.001106,1453,1453,1453,0,0,0,0,1.0


symbol,asset_type,provider,coverage_pct,last_error
AXIA,CEDEAR,byma_open_data,1.0,HTTP 400
EAD,CEDEAR,null,0.0,HTTP 400
LND,CEDEAR,byma_open_data,1.0,HTTP 400
MDLZD,CEDEAR,null,0.0,HTTP 400
TEFO,CEDEAR,byma_open_data,1.0,HTTP 400


## 5. Comparación contra el benchmark y calibración de τ
τ es la mitad de la mediana del rango diario de cada tipo (ADR-009). `share_above` no es sobreprecio: incluye el azar intradiario. La consulta por cliente es de la etapa 08.

In [0]:
mapping = spark.table(mapping_table)
display(quotes.daily_range_by_type(spark.table(market_table).where("is_active"), mapping))
display(quotes.deviation_summary(matched, mapping))

asset_type,quote_days,median_range,p90_range
ACCION_LOCAL,5035,0.051,0.0862
BONO_SOBERANO,3387,0.0143,0.0332
BONO_SUBSOBERANO,406,0.0251,0.049
CEDEAR,31014,0.0291,0.0681
LETRA_LECAP_BONCAP,1070,0.0109,0.0253
ON,1369,0.0114,0.0442
WARRANT,105,0.0789,0.1746


asset_type,trades,median_deviation,median_abs,p90_abs,share_above,share_above_high,share_below_low
CEDEAR,53232,0.001,0.0102,0.0411,0.2151,0.11,0.0822
ACCION_LOCAL,18092,0.0015,0.0157,0.0512,0.1757,0.08,0.0716
BONO_SOBERANO,17656,-3.0E-4,0.0042,0.0125,0.1217,0.1424,0.1458
LETRA_LECAP_BONCAP,4367,4.0E-4,0.0028,0.0078,0.1145,0.1795,0.1983
BONO_SUBSOBERANO,215,0.0011,0.005,0.0223,0.186,0.2279,0.1256
ON,104,0.001,0.0048,0.0273,0.1827,0.3173,0.0481
WARRANT,95,0.0191,0.0496,0.1719,0.3895,0.1158,0.0526


In [0]:
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "pipeline_run_id": RUN_ID, "rows_out": matched_count, "coverage_pct": overall["coverage_pct"], "fetched": fetched}))